# SIS 2 — Pastebin: упрощённый Step 5

Самостоятельная версия: сохранить текст, получить ссылку, прочитать его через REST/gRPC, проверить надёжность и сравнить протоколы. Исходный ноутбук для запуска не нужен.

**Сохранено:** неизменяемые записи, SQLite, лимит 1 МиБ, TTL 60 секунд–30 дней, идемпотентность, REST/JSON и gRPC/Protobuf, функциональные проверки, повторы с backoff/jitter и бюджетом, тест дедлайна, бенчмарк p50/p95/p99 и размеров тел, CSV и график. Убраны неиспользуемые импорты, pandas, PDF и сложное оформление. Длинные части разделены по задачам.

**По заданию требуется Java; здесь Python-прототип.** Это отклонение нужно объяснить. REST служит исходной базой, gRPC — вторым транспортом для сравнения на общих правилах и базе. Localhost/SQLite не доказывают production RPS/p99.

Запускай сверху вниз в Python 3.11+. Не запускай одновременно с исходной версией: порты одинаковые. `Run All` по умолчанию останавливает серверы в конце. Для браузера выставь `STOP_AT_END=False` в настройках. Данные этой версии сохраняются отдельно: `sis2_pastebin_runtime/simplified/`.


## 1. Зависимости


In [ ]:
# Если пакеты ещё не установлены, убери # и выполни команду:
# %pip install grpcio grpcio-tools protobuf fastapi uvicorn httpx pydantic matplotlib


### Объяснение

Команда закомментирована: готовое окружение не нужно менять при каждом запуске. `%pip` устанавливает библиотеки в выбранное ядро. FastAPI/Uvicorn обслуживают REST, gRPC/Protobuf — RPC, HTTPX отправляет HTTP-запросы, Matplotlib строит график. pandas не нужен.

**Вопрос преподавателя:** почему важно окружение? Версии Python и библиотек влияют на воспроизводимость результатов; терминальный pip может относиться к другому Python.


## 2. Настройки


In [ ]:
from pathlib import Path
from contextlib import contextmanager
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor
import csv
import hashlib
import json
import math
import platform
import random
import secrets
import sqlite3
import statistics
import string
import subprocess
import sys
import threading
import time
import uuid

import grpc
import httpx
import uvicorn
from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import PlainTextResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel

WORKDIR = Path.cwd() / 'sis2_pastebin_runtime' / 'simplified'
WORKDIR.mkdir(parents=True, exist_ok=True)
DB_PATH = WORKDIR / 'pastebin.sqlite3'
REST_PORT, GRPC_PORT = 18080, 50071
BASE_URL = f'http://127.0.0.1:{REST_PORT}'
MAX_BYTES = 1_048_576
MIN_TTL, MAX_TTL, DEFAULT_TTL = 60, 30 * 86400, 7 * 86400
STOP_AT_END = True


### Объяснение

Настройки находятся в одном месте. `WORKDIR` отделяет данные новой версии, `DB_PATH` указывает базу. Размер считается в байтах UTF-8, TTL — в секундах. `86400` — сутки.

`STOP_AT_END=True` завершает полный запуск остановкой. Для просмотра через Safari поставь `False`, а по окончании вызови `stop_servers()`.

**Вопрос:** порты случайные? Они явно заданы, должны быть свободны и согласованы с клиентом. `127.0.0.1` — адрес только этого компьютера; ссылка не публикуется в интернете.


## 3. Контракт gRPC


In [ ]:
PROTO = r'''syntax = "proto3";
package pastebin;
service Pastebin {
  rpc CreateSnippet(CreateSnippetRequest) returns (CreateSnippetResponse);
  rpc GetSnippet(GetSnippetRequest) returns (GetSnippetResponse);
}
message CreateSnippetRequest {
  string content = 1;
  string title = 2;
  string language = 3;
  int32 ttl_seconds = 4;
  string idempotency_key = 5;
}
message CreateSnippetResponse {
  string id = 1;
  string url = 2;
  string created_at = 3;
  string expires_at = 4;
}
message GetSnippetRequest { string id = 1; }
message GetSnippetResponse {
  string id = 1;
  string content = 2;
  string title = 3;
  string language = 4;
  string created_at = 5;
  string expires_at = 6;
}
'''
(WORKDIR / "pastebin.proto").write_text(PROTO, encoding="utf-8")
subprocess.run([sys.executable, "-m", "grpc_tools.protoc", f"-I{WORKDIR}", f"--python_out={WORKDIR}", f"--grpc_python_out={WORKDIR}", str(WORKDIR / "pastebin.proto")], check=True)
sys.path.insert(0, str(WORKDIR))
import pastebin_pb2 as pb
import pastebin_pb2_grpc as pb_grpc
print("Protocol Buffers stubs generated in", WORKDIR)

### Объяснение

`PROTO` хранит описание методов и сообщений, а не ответ. `CreateSnippet` принимает текст и параметры, возвращает ID, URL и даты. `GetSnippet` принимает ID, возвращает текст и метаданные. Числа полей — номера в Protobuf, не значения по умолчанию.

Компилятор `grpc_tools.protoc` создаёт `pb` с классами сообщений и `pb_grpc` с серверной основой и клиентским stub. `sys.executable` выбирает Python текущего ядра, `check=True` обнаруживает ошибку компиляции.

**Вопрос:** где настоящий ответ? В переменной `created`, полученной вызовом клиента. Генерация кода не реализует сохранение текста — его выполняет сервис ниже.


## 4. Соединение и таблицы SQLite


In [ ]:
@contextmanager
def connect_database(path):
    db = sqlite3.connect(path, timeout=5)
    db.row_factory = sqlite3.Row
    try:
        with db:
            yield db
    finally:
        db.close()

with connect_database(DB_PATH) as db:
    db.execute("PRAGMA journal_mode=WAL")
    db.execute("""CREATE TABLE IF NOT EXISTS snippets (
        id TEXT PRIMARY KEY, content TEXT NOT NULL, title TEXT,
        language TEXT NOT NULL, size_bytes INTEGER NOT NULL,
        created_at REAL NOT NULL, expires_at REAL NOT NULL)""")
    db.execute("CREATE INDEX IF NOT EXISTS snippets_expires_idx ON snippets(expires_at)")
    db.execute("""CREATE TABLE IF NOT EXISTS idempotency_keys (
        key TEXT PRIMARY KEY, fingerprint TEXT NOT NULL,
        response_json TEXT NOT NULL, created_at REAL NOT NULL)""")


### Объяснение

**`connect_database()`** открывает соединение, отдаёт его через `yield`, фиксирует транзакцию при успехе/откатывает при исключении и обязательно закрывает соединение в `finally`. Декоратор `@contextmanager` позволяет использовать функцию в `with`.

`snippets` хранит тексты, `idempotency_keys` — ключи запросов, отпечатки и ответы. `PRIMARY KEY` запрещает дубликаты, индекс помогает искать по TTL. WAL улучшает совместную работу чтения и записи, но не создаёт неограниченный параллелизм.

**Вопрос:** почему явно закрывается соединение? Контекст SQLite управляет транзакцией, но сам соединение не закрывает. Эта версия устраняет такой недостаток исходного кода.


## 5. Общий сервис


In [ ]:
class ServiceError(Exception):
    def __init__(self, status, message):
        super().__init__(message)
        self.status, self.message = status, message

class PastebinService:
    def __init__(self, db_path):
        self.db_path = str(db_path)
        self._lock = threading.RLock()

    def _connect(self):
        return connect_database(self.db_path)

    @staticmethod
    def _iso(ts):
        return datetime.fromtimestamp(ts, timezone.utc).isoformat().replace('+00:00', 'Z')

    def _validate(self, content, title, language, ttl_seconds, key):
        if not isinstance(content, str):
            raise ServiceError(422, 'content must be a string')
        size = len(content.encode('utf-8'))
        if size > MAX_BYTES:
            raise ServiceError(413, 'content exceeds 1 MiB')
        if size < 1:
            raise ServiceError(422, 'content must not be empty')
        if not isinstance(ttl_seconds, int) or not MIN_TTL <= ttl_seconds <= MAX_TTL:
            raise ServiceError(422, 'ttl_seconds out of range')
        if not isinstance(title, str) or len(title) > 200:
            raise ServiceError(422, 'invalid title')
        if not isinstance(language, str) or not 1 <= len(language) <= 32:
            raise ServiceError(422, 'invalid language')
        if key is not None:
            try: key = str(uuid.UUID(key))
            except (ValueError, TypeError, AttributeError):
                raise ServiceError(422, 'idempotency key must be a UUID')
        return size, key

    def create(self, content, title='', language='plaintext', ttl_seconds=DEFAULT_TTL, key=None):
        size, key = self._validate(content, title, language, ttl_seconds, key)
        payload = dict(content=content, title=title, language=language, ttl_seconds=ttl_seconds)
        fingerprint = hashlib.sha256(json.dumps(payload, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
        with self._lock, self._connect() as db:
            db.execute('BEGIN IMMEDIATE')
            if key:
                old = db.execute('SELECT fingerprint,response_json FROM idempotency_keys WHERE key=?', (key,)).fetchone()
                if old:
                    if old['fingerprint'] != fingerprint:
                        raise ServiceError(409, 'idempotency key reused with different content')
                    return json.loads(old['response_json'])
            now = time.time()
            for _ in range(3):
                snippet_id = ''.join(secrets.choice(string.ascii_letters + string.digits) for _ in range(8))
                try:
                    db.execute('INSERT INTO snippets VALUES (?,?,?,?,?,?,?)',
                               (snippet_id, content, title, language, size, now, now+ttl_seconds))
                    break
                except sqlite3.IntegrityError:
                    continue
            else:
                raise ServiceError(503, 'unable to allocate unique ID')
            response = dict(id=snippet_id, url=f'{BASE_URL}/s/{snippet_id}',
                            created_at=self._iso(now), expires_at=self._iso(now+ttl_seconds))
            if key:
                db.execute('INSERT INTO idempotency_keys VALUES (?,?,?,?)',
                           (key, fingerprint, json.dumps(response), now))
            db.commit()
            return response

    def get(self, snippet_id):
        with self._connect() as db:
            row = db.execute('SELECT * FROM snippets WHERE id=? AND expires_at>?',
                             (snippet_id, time.time())).fetchone()
        if row is None:
            raise ServiceError(404, 'snippet not found or expired')
        return dict(id=row['id'], content=row['content'], title=row['title'],
                    language=row['language'], created_at=self._iso(row['created_at']),
                    expires_at=self._iso(row['expires_at']))

    def cleanup(self):
        with self._lock, self._connect() as db:
            result = db.execute('DELETE FROM snippets WHERE id IN (SELECT id FROM snippets WHERE expires_at <= ? LIMIT 1000)', (time.time(),))
            db.commit()
            return result.rowcount

service = PastebinService(DB_PATH)
print('Storage ready:', DB_PATH)

### Объяснение

Это главный класс. Длинный метод создания разделён на проверку входа и запись; SQL и транзакции сохранены явно.

**`_validate()`** проверяет текст, UTF-8-размер, TTL, заголовок, язык и UUID. Возвращает размер и нормализованный ключ. **`create()`** вычисляет отпечаток параметров, начинает `BEGIN IMMEDIATE`, ищет ключ и либо возвращает прежний ответ, либо сохраняет сниппет и ответ одной транзакцией. ID генерируется из восьми букв/цифр с тремя попытками при совпадении. SQL-параметры `?` передают данные отдельно от команды.

**`get()`** читает только при `expires_at > сейчас`, иначе даёт 404. **`cleanup()`** удаляет до 1000 истёкших строк за вызов, без автоматического расписания. **`_iso()`** форматирует UTC-время. Пользовательского обновления текста нет.

**Вопросы:** SHA-256 шифрует текст? Нет, это отпечаток для сравнения параметров. Что даёт повтор с одним ключом? Тот же ответ при совпадении, 409 при различии. Зачем RLock и транзакция? Первый согласует потоки одного объекта, вторая и ограничения защищают базу. Почему истёкший текст остаётся в базе? Доступность и физическая очистка разделены. Ключи не очищаются автоматически: старый повтор не продлевает TTL.


## 6. Обработчики REST


In [ ]:
app = FastAPI(title='SIS2 Pastebin REST baseline')

class CreateBody(BaseModel):
    content: str
    title: str = ''
    language: str = 'plaintext'
    ttl_seconds: int = DEFAULT_TTL

def rest_call(fn, *args, **kwargs):
    try: return fn(*args, **kwargs)
    except ServiceError as e: raise HTTPException(status_code=e.status, detail=e.message)

@app.post('/v1/snippets', status_code=201)
def create_rest(body: CreateBody, idempotency_key: str | None = Header(default=None)):
    return rest_call(service.create, body.content, body.title, body.language, body.ttl_seconds, idempotency_key)

@app.get('/v1/snippets/{snippet_id}')
def get_rest(snippet_id: str):
    return rest_call(service.get, snippet_id)

@app.get('/s/{snippet_id}', response_class=PlainTextResponse)
def get_public(snippet_id: str):
    return rest_call(service.get, snippet_id)['content']



### Объяснение

`CreateBody` — структура JSON и значения по умолчанию. Pydantic проверяет форму запроса, общие бизнес-правила проверяет сервис. **`create_rest()`** обрабатывает POST и возвращает 201; **`get_rest()`** возвращает JSON; **`get_public()`** — только текст. **`rest_call()`** переводит `ServiceError` в HTTP-ошибку с `detail`.

`Header` читает `Idempotency-Key`. URL содержит ID, не заголовок текста. Любое чтение проходит один `get()` и проверку TTL.

**Вопрос:** `/docs` — это наш сайт? Это автоматически созданный интерфейс API: он отправляет реальные запросы. Отдельной главной страницы `/` и списка записей здесь нет.


## 7. Обработчики gRPC


In [ ]:
GRPC_ERROR_CODES = {
    404: grpc.StatusCode.NOT_FOUND,
    409: grpc.StatusCode.ALREADY_EXISTS,
    413: grpc.StatusCode.RESOURCE_EXHAUSTED,
    422: grpc.StatusCode.INVALID_ARGUMENT,
    503: grpc.StatusCode.UNAVAILABLE,
}

class PastebinGrpc(pb_grpc.PastebinServicer):
    def CreateSnippet(self, request, context):
        try:
            result = service.create(request.content, request.title, request.language or 'plaintext',
                                    request.ttl_seconds or DEFAULT_TTL, request.idempotency_key or None)
            return pb.CreateSnippetResponse(**result)
        except ServiceError as e:
            context.abort(GRPC_ERROR_CODES.get(e.status, grpc.StatusCode.INTERNAL), e.message)

    def GetSnippet(self, request, context):
        try:
            return pb.GetSnippetResponse(**service.get(request.id))
        except ServiceError as e:
            context.abort(GRPC_ERROR_CODES.get(e.status, grpc.StatusCode.INTERNAL), e.message)



### Объяснение

**`CreateSnippet()` и `GetSnippet()`** берут поля сообщений и вызывают тот же `service`. `**result` превращает словарь в именованные аргументы сообщения ответа. `context.abort()` завершает RPC с gRPC-статусом из таблицы.

`or` заменяет пустой язык и нулевой TTL значениями по умолчанию. Это сохранённый нюанс исходника: REST с явным TTL=0 получает 422, gRPC выбирает семь дней. Поэтому все возможные входы не полностью эквивалентны, хотя допустимые одинаковые параметры проходят общий сервис.

**Вопрос:** как запись gRPC читается через HTTP? База и логика общие; ответ содержит публичный REST-URL.


## 8. Запуск и остановка


In [ ]:
rest_server = None
rest_thread = None
grpc_server = None
channel = None
http_client = None

def stop_servers():
    global rest_server, rest_thread, grpc_server, channel, http_client
    if channel is not None:
        channel.close()
        channel = None
    if http_client is not None:
        http_client.close()
        http_client = None
    if grpc_server is not None:
        if not grpc_server.stop(grace=1).wait(timeout=3):
            raise RuntimeError('gRPC server did not stop')
        grpc_server = None
    if rest_server is not None:
        rest_server.should_exit = True
    if rest_thread is not None:
        rest_thread.join(timeout=5)
        if rest_thread.is_alive():
            raise RuntimeError('REST server did not stop')
    rest_server = rest_thread = None

def start_servers():
    global rest_server, rest_thread, grpc_server, channel, http_client
    if rest_thread is not None and rest_thread.is_alive():
        raise RuntimeError('Call stop_servers() before starting again')
    try:
        grpc_server = grpc.server(ThreadPoolExecutor(max_workers=10), options=[
            ('grpc.max_receive_message_length', 2 * MAX_BYTES),
            ('grpc.max_send_message_length', 2 * MAX_BYTES)])
        pb_grpc.add_PastebinServicer_to_server(PastebinGrpc(), grpc_server)
        if grpc_server.add_insecure_port(f'127.0.0.1:{GRPC_PORT}') == 0:
            raise RuntimeError('gRPC port is busy')
        grpc_server.start()
        config = uvicorn.Config(app, host='127.0.0.1', port=REST_PORT,
                                log_level='warning', access_log=False)
        rest_server = uvicorn.Server(config)
        rest_thread = threading.Thread(target=rest_server.run, daemon=True)
        rest_thread.start()
        for _ in range(100):
            if rest_server.started:
                break
            if not rest_thread.is_alive():
                raise RuntimeError('REST did not start; check whether the port is busy')
            time.sleep(0.05)
        else:
            raise RuntimeError('REST startup timed out')
        channel = grpc.insecure_channel(f'127.0.0.1:{GRPC_PORT}')
        grpc.channel_ready_future(channel).result(timeout=5)
        http_client = httpx.Client(timeout=5)
    except Exception:
        stop_servers()
        raise
    print('REST documentation:', BASE_URL + '/docs')
    print('gRPC port:', GRPC_PORT)

start_servers()
stub = pb_grpc.PastebinStub(channel)


### Объяснение

**`start_servers()`** запускает серверы, ждёт готовности и создаёт клиентов. При ошибке очищает ресурсы. **`stop_servers()`** закрывает клиентов, сигнализирует серверам и ждёт остановки. Отсутствующие объекты пропускаются; повторная остановка допустима.

Транспортный лимит gRPC — 2 МиБ с учётом полей, но текст ограничен 1 МиБ в сервисе. Серверы работают фоном, поэтому ячейка заканчивается, а запросы обслуживаются дальше.

Не выполняй всю эту ячейку повторно поверх работающих серверов: первые строки сбрасывают ссылки. Используй функции или перезапуск ядра. После остановки для нового старта нужны `start_servers()` и `stub = pb_grpc.PastebinStub(channel)`.

**Вопрос:** `address already in use` — что делать? Остановить прежний запуск. Функция этого ядра не управляет внешним сервером в другом процессе.


## 9. REST-клиент и пример


In [ ]:
def rest_request(method, path, *, client=None, **kwargs):
    client = client if client is not None else http_client
    response = client.request(method, BASE_URL + path, **kwargs)
    response.raise_for_status()
    return response.json()

payload = dict(content='Привет из REST', title='demo', language='plaintext', ttl_seconds=3600)
headers = {'Idempotency-Key': str(uuid.uuid4())}
created_rest = rest_request('POST', '/v1/snippets', json=payload, headers=headers)
repeated_rest = rest_request('POST', '/v1/snippets', json=payload, headers=headers)
assert created_rest == repeated_rest
read_rest = rest_request('GET', '/v1/snippets/' + created_rest['id'])
assert read_rest['content'] == payload['content']
public = http_client.get(created_rest['url'])
assert public.status_code == 200 and public.text == payload['content']
print('PASS: REST create, read, repeat, public URL')
print('Открой:', created_rest['url'])


### Объяснение

**`rest_request()`** объединяет отправку, проверку статуса и разбор JSON. Общий клиент переиспользует соединения; отдельный клиент можно подставить для теста.

Пример создаёт запись, повторяет тот же запрос с одним ключом, читает JSON и публичный текст. `assert` проверяет результат, `raise_for_status()` превращает HTTP-ошибку в исключение.

**Вопрос:** код из content выполнится? Нет, это сохраняемый текст. Как получить ссылку? Она уже в `created_rest['url']`. Без ключа каждый POST создаёт новую запись.


## 10. gRPC-клиент и пример


In [ ]:
key = str(uuid.uuid4())
request = pb.CreateSnippetRequest(content='print("Hello from SIS2")', title='demo',
                                  language='python', ttl_seconds=3600, idempotency_key=key)
created = stub.CreateSnippet(request, timeout=2)
created_again = stub.CreateSnippet(request, timeout=2)
assert created.id == created_again.id, 'idempotency failed'
read_grpc = stub.GetSnippet(pb.GetSnippetRequest(id=created.id), timeout=2)
assert read_grpc.content == request.content

assert rest_request('GET', '/v1/snippets/' + created.id)['content'] == request.content
print('PASS: gRPC create, read, repeat, cross-protocol read')
print('Открой:', created.url)


### Объяснение

`stub.CreateSnippet()` отправляет сообщение `pb`, ответ хранится в `created`. Повтор должен вернуть тот же ID. `GetSnippet` проверяет исходный текст; дополнительное чтение через REST показывает общую базу.

**Вопрос:** это автоматический retry? Пока нет: два вызова написаны явно. `timeout=2` ограничивает ожидание RPC, но обёртка повторов появится ниже. `PROTO` описывает структуру, а `created` содержит конкретный ответ.


## 11. Ошибочные REST-запросы


In [ ]:
with TestClient(app) as client:
    assert client.get('/v1/snippets/NOTFOUND').status_code == 404
    assert client.post('/v1/snippets', json={'content': '', 'ttl_seconds': 60}).status_code == 422
    assert client.post('/v1/snippets', json={'content': 'a' * (MAX_BYTES + 1), 'ttl_seconds': 60}).status_code == 413
    conflict_key = str(uuid.uuid4())
    h = {'Idempotency-Key': conflict_key}
    a = client.post('/v1/snippets', json={'content': 'first', 'ttl_seconds': 60}, headers=h)
    b = client.post('/v1/snippets', json={'content': 'different', 'ttl_seconds': 60}, headers=h)
    assert a.status_code == 201 and b.status_code == 409
print('PASS: 404, empty content, oversize content, conflicting idempotency key')

### Объяснение

Проверяются 404 для неизвестного ID, 422 для пустого текста, 413 для превышения размера и 409 для изменения данных с прежним ключом. Первый запрос с ключом должен вернуть 201.

`TestClient` вызывает приложение без сети, но с реальными сервисом и SQLite. ASCII-строка предсказуемо занимает один байт на символ.

**Вопрос:** это нагрузочный тест? Нет, это проверки корректности. Они дополняют сетевые примеры. Далее проверяются допустимые границы и истечение TTL.


## 12. Границы и истечение TTL


In [ ]:
def expect_service_error(status, **data):
    try:
        service.create(**data)
    except ServiceError as error:
        assert error.status == status
    else:
        raise AssertionError(f'Expected service error {status}')

for ttl in (MIN_TTL, MAX_TTL):
    assert service.create('TTL boundary', ttl_seconds=ttl)['id']
for ttl in (MIN_TTL - 1, MAX_TTL + 1):
    expect_service_error(422, content='test', ttl_seconds=ttl)
boundary = service.create('a' * MAX_BYTES)
assert len(service.get(boundary['id'])['content'].encode('utf-8')) == MAX_BYTES
expect_service_error(413, content='я' * (MAX_BYTES // 2 + 1))
expect_service_error(422, content='test', key='not-a-uuid')
key = str(uuid.uuid4())
first = service.create('idempotency test', key=key)
assert first == service.create('idempotency test', key=key)
expect_service_error(409, content='changed', key=key)
with service._connect() as db:
    assert db.execute('SELECT COUNT(*) FROM snippets WHERE id=?', (first['id'],)).fetchone()[0] == 1
    db.execute('UPDATE snippets SET expires_at=? WHERE id=?', (time.time() - 1, first['id']))
try:
    service.get(first['id'])
except ServiceError as error:
    assert error.status == 404
else:
    raise AssertionError('Expired snippet was returned')
assert service.cleanup() >= 1
print('PASS: size / TTL boundaries, UUID, idempotency, expiry and cleanup')


### Объяснение

Проверяются оба допустимых края TTL и значения за пределами, ровно 1 МиБ и превышение байтового размера кириллицей, некорректный UUID. Повтор должен вернуть тот же ответ, а SQL-подсчёт — одну строку.

Чтобы не ждать минимум минуту, тест переводит срок **своей тестовой строки** в прошлое через SQL. Пользовательского изменения текста в API не появляется. Проверяются отказ чтения и очистка.

**Вопрос:** доказывает ли это все конкурентные случаи? Нет: это последовательный тест. Что важнее при TTL — удаление или недоступность? Немедленная недоступность, а физическая очистка может быть позже.


## 13. Алгоритм повторов


In [ ]:
from dataclasses import dataclass

RETRYABLE_HTTP = {429, 503, 504}
RETRYABLE_GRPC = {grpc.StatusCode.UNAVAILABLE, grpc.StatusCode.RESOURCE_EXHAUSTED,
                  grpc.StatusCode.DEADLINE_EXCEEDED}

@dataclass
class RetryStats:
    attempts: int = 0
    elapsed_ms: float = 0.0

def call_with_retries(operation, retryable, *, budget_s=0.140, max_attempts=3,
                      base_backoff_s=0.010, rng=None, sleep=time.sleep, clock=time.monotonic):
    """operation(remaining_seconds) -> result; never start an attempt after deadline."""
    if budget_s <= 0 or max_attempts < 1:
        raise ValueError('budget_s must be positive; max_attempts >= 1')
    rng = rng or random.Random()
    start = clock()
    deadline = start + budget_s
    stats = RetryStats()
    for attempt in range(max_attempts):
        remaining = deadline - clock()
        if remaining <= 0:
            raise TimeoutError('overall operation deadline exceeded')
        stats.attempts += 1
        try:
            value = operation(remaining)
            stats.elapsed_ms = (clock() - start) * 1000
            return value, stats
        except Exception as exc:
            if not retryable(exc) or attempt + 1 >= max_attempts:
                raise
            remaining = deadline - clock()
            if remaining <= 0:
                raise TimeoutError('overall operation deadline exceeded') from exc
            backoff = base_backoff_s * (2 ** attempt) * (0.5 + rng.random())
            # Do not spend the entire remaining budget sleeping.
            delay = min(backoff, remaining / 2)
            if delay > 0:
                sleep(delay)
    raise AssertionError('unreachable')

def grpc_retryable(exc):
    return isinstance(exc, grpc.RpcError) and exc.code() in RETRYABLE_GRPC

def http_retryable(exc):
    return (isinstance(exc, httpx.HTTPStatusError) and
            exc.response.status_code in RETRYABLE_HTTP) or isinstance(
            exc, (httpx.TimeoutException, httpx.TransportError))



### Объяснение

**`call_with_retries()`** принимает операцию и проверку повторяемости ошибки. По умолчанию общий бюджет 140 мс и максимум три попытки всего. Перед каждым вызовом считает остаток через `monotonic()` и передаёт транспорту.

Backoff увеличивает паузу экспоненциально, jitter добавляет случайность. При неповторяемой ошибке или последней попытке исключение выходит наружу. `RetryStats` возвращает число попыток и время успешной операции. Для тестов можно подставить часы, sleep и генератор случайности.

Ограничение сохранено: алгоритм не начинает новую попытку после дедлайна, но не прерывает произвольную функцию, игнорирующую таймаут. HTTPX ограничивает сетевые фазы, не всю операцию строго одним таймером. `RESOURCE_EXHAUSTED` может означать и постоянное превышение размера; повтор в таком случае бесполезен.

**Вопрос:** почему не три отдельных таймаута по 140 мс? Они превысят общий бюджет. Почему не повторять 422? Данные не исправятся от ожидания.


## 14. Надёжные клиенты


In [ ]:
def create_grpc_reliable(content, *, title='', language='plaintext', ttl_seconds=3600,
                         key=None, budget_s=0.140, grpc_stub=None):
    request = pb.CreateSnippetRequest(content=content, title=title, language=language,
        ttl_seconds=ttl_seconds, idempotency_key=key or str(uuid.uuid4()))
    active_stub = grpc_stub if grpc_stub is not None else stub
    return call_with_retries(
        lambda remaining: active_stub.CreateSnippet(request, timeout=remaining),
        grpc_retryable, budget_s=budget_s)

def get_grpc_reliable(snippet_id, *, budget_s=0.140, grpc_stub=None):
    active_stub = grpc_stub if grpc_stub is not None else stub
    request = pb.GetSnippetRequest(id=snippet_id)
    return call_with_retries(
        lambda remaining: active_stub.GetSnippet(request, timeout=remaining),
        grpc_retryable, budget_s=budget_s)

def create_rest_reliable(content, *, title='', language='plaintext', ttl_seconds=3600,
                         key=None, budget_s=0.140, client=None):
    payload = dict(content=content, title=title, language=language, ttl_seconds=ttl_seconds)
    headers = {'Idempotency-Key': key or str(uuid.uuid4())}
    return call_with_retries(
        lambda remaining: rest_request('POST', '/v1/snippets', client=client,
            json=payload, headers=headers, timeout=remaining),
        http_retryable, budget_s=budget_s)

def get_rest_reliable(snippet_id, *, budget_s=0.140, client=None):
    return call_with_retries(
        lambda remaining: rest_request('GET', '/v1/snippets/' + snippet_id,
            client=client, timeout=remaining), http_retryable, budget_s=budget_s)


### Объяснение

Четыре обёртки создают/читают через оба транспорта. Повторяющийся HTTP-код заменён вызовом `rest_request()`. `lambda remaining` — короткая функция, принимающая остаток бюджета.

**Главное:** ключ и тело создаются до повторов и не меняются между попытками. **Вопрос:** если ответ потерян, запись точно не сохранена? Нет, сервер мог уже сохранить её. Новый UUID при повторе создаст дубликат, прежний ключ вернёт прежний ответ.


## 15. Надёжные клиенты на реальных серверах


### Объяснение

Обёртки проверяются на настоящих серверах: создание и чтение по обоим протоколам. Кортеж распаковывается в результат и статистику, содержимое сверяется.

Здесь бюджет явно увеличен до двух секунд для устойчивого успешного сценария. **Вопрос:** проверяет ли успех восстановление после сбоя? Нет, искусственные ошибки проверяются отдельно следующей ячейкой.


## 16. Управляемые HTTP-сбои


In [ ]:
# 9c. HTTP retry injection: 503 twice, then 201; confirm SAME key and payload.
seen = []
def fake_handler(request):
    seen.append((request.headers.get('Idempotency-Key'), request.content))
    if len(seen) < 3:
        return httpx.Response(503, json={'detail': 'temporarily unavailable'})
    return httpx.Response(201, json={'id': 'test1234', 'url': 'http://example/s/test1234'})
with httpx.Client(transport=httpx.MockTransport(fake_handler)) as fake_client:
    result, stats = create_rest_reliable('retry example', key=str(uuid.uuid4()),
                                        budget_s=2, client=fake_client)
assert stats.attempts == 3 and len(set(k for k, _ in seen)) == 1
assert len(set(body for _, body in seen)) == 1
assert result['id'] == 'test1234'
print('PASS: retry 503 twice then success, key and body unchanged')

# Non-retryable HTTP 422 must stop after one attempt.
calls = []
def reject(request):
    calls.append(1)
    return httpx.Response(422, json={'detail': 'bad request'})
with httpx.Client(transport=httpx.MockTransport(reject)) as bad_client:
    try:
        create_rest_reliable('bad', budget_s=2, client=bad_client)
        raise AssertionError('Expected HTTP 422')
    except httpx.HTTPStatusError as exc:
        assert exc.response.status_code == 422
assert len(calls) == 1
print('PASS: validation errors are not retried')

### Объяснение

`MockTransport` вместо сети вызывает Python-обработчик: два ответа 503, потом 201. Проверяются ровно три попытки и одинаковые ключ/байты тела. Затем всегда возвращается 422 и требуется одна попытка.

**Вопрос:** запись `test1234` появилась в базе? Нет, это фиктивный ответ. Макет точно проверяет политику повторов, но не реальную сеть и не транзакции; поэтому рядом есть настоящие интеграционные проверки.


## 17. Реальный дедлайн gRPC


In [ ]:
# 9d. Actual gRPC deadline: temporary server sleeps longer than allowed.
class SlowPastebin(pb_grpc.PastebinServicer):
    def GetSnippet(self, request, context):
        time.sleep(0.25)
        return pb.GetSnippetResponse(id=request.id, content='late')
    def CreateSnippet(self, request, context):
        time.sleep(0.25)
        return pb.CreateSnippetResponse(id='late')

slow_server = grpc.server(ThreadPoolExecutor(max_workers=4))
pb_grpc.add_PastebinServicer_to_server(SlowPastebin(), slow_server)
slow_port = slow_server.add_insecure_port('127.0.0.1:0')
assert slow_port > 0
slow_server.start()
slow_channel = grpc.insecure_channel(f'127.0.0.1:{slow_port}')
try:
    slow_stub = pb_grpc.PastebinStub(slow_channel)
    start = time.monotonic()
    try:
        get_grpc_reliable('anything', budget_s=0.140, grpc_stub=slow_stub)
        raise AssertionError('Expected deadline exceeded')
    except (grpc.RpcError, TimeoutError) as exc:
        if isinstance(exc, grpc.RpcError):
            assert exc.code() == grpc.StatusCode.DEADLINE_EXCEEDED
    elapsed = time.monotonic() - start
    assert elapsed < 0.5, f'Unexpectedly slow deadline enforcement: {elapsed:.3f}s'
    print(f'PASS: gRPC deadline enforced (elapsed {elapsed*1000:.1f} ms)')
finally:
    slow_channel.close()
    slow_server.stop(grace=0).wait(timeout=2)

### Объяснение

Временный сервер отвечает через 250 мс, клиент ждёт с бюджетом 140 мс. Порт `0` выбирается ОС. Ожидается `DEADLINE_EXCEEDED` или `TimeoutError`; предел проверки 500 мс допускает накладные расходы, а не обещает точный SLA.

`finally` закрывает канал и сервер даже при ошибке. **Вопрос:** обработчик остановился сразу с клиентом? Не обязательно: здесь он продолжает sleep. Поэтому таймаут не доказывает откат побочных эффектов.


## 18. Бенчмарк: настройки и статистика


In [ ]:
# Start with a small repeatable test. Change to (100, 1000) for more samples.
BENCH_REQUESTS_PER_CASE = 100
BENCH_WARMUP = 10
BENCH_SIZES = (100, 1024, 10240)
BENCH_TIMEOUT_S = 5.0


def percentile(values, p):
    if not values:
        return float('nan')
    ordered = sorted(values)
    pos = (len(ordered) - 1) * p / 100
    lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] + (ordered[hi] - ordered[lo]) * (pos - lo)


def summarize(protocol, operation, size, times_ms, failures, request_bytes, response_bytes):
    return {
        'protocol': protocol, 'operation': operation, 'content_bytes': size,
        'attempts': len(times_ms) + failures, 'successes': len(times_ms),
        'failures': failures,
        'success_rate_pct': round(100 * len(times_ms) / (len(times_ms) + failures), 2),
        'mean_ms': round(statistics.mean(times_ms), 3) if times_ms else None,
        'p50_ms': round(percentile(times_ms, 50), 3) if times_ms else None,
        'p95_ms': round(percentile(times_ms, 95), 3) if times_ms else None,
        'p99_ms': round(percentile(times_ms, 99), 3) if times_ms else None,
        'request_bytes': request_bytes, 'response_bytes': response_bytes,
    }

print('Environment:', platform.platform(), '| Python:', platform.python_version())
print('Benchmark:', BENCH_REQUESTS_PER_CASE, 'measured calls per protocol / operation / size')

### Объяснение

Сохранены 100 измеряемых запросов и 10 прогревочных, размеры 100/1024/10240 байт, два протокола и две операции. Это 12 сценариев и 1200 измеряемых попыток; прогрев и подготовительные записи не входят в них.

**`percentile()`** сортирует и линейно интерполирует значения: p50 — медиана, p95/p99 — оценка хвоста. **`summarize()`** собирает успехи/ошибки, среднее, перцентили и размеры. Задержки только успешных запросов; при отсутствии успеха — `None`.

**Вопрос:** почему нужен failures рядом с p95? Быстрые успешные ответы могут скрыть много провалов. p99 из 100 измерений нестабилен. Для большей выборки меняй `BENCH_REQUESTS_PER_CASE` на число, например `1000`, не на кортеж.


## 19. Один измеряемый запрос


In [ ]:
def measure_request(protocol, operation, content, snippet_id=None):
    creating = operation == 'Create'
    payload = dict(content=content, title='benchmark', language='plaintext', ttl_seconds=3600)
    key = str(uuid.uuid4()) if creating else None
    if protocol == 'gRPC':
        request = (pb.CreateSnippetRequest(**payload, idempotency_key=key) if creating
                   else pb.GetSnippetRequest(id=snippet_id))
        request_bytes = len(request.SerializeToString())
        rpc = stub.CreateSnippet if creating else stub.GetSnippet
        start = time.perf_counter_ns()
        response = rpc(request, timeout=BENCH_TIMEOUT_S)
        elapsed_ms = (time.perf_counter_ns() - start) / 1e6
        response_bytes = len(response.SerializeToString())
        assert response.id if creating else response.content == content
    else:
        path = '/v1/snippets' if creating else '/v1/snippets/' + snippet_id
        request_bytes = len(json.dumps(payload, ensure_ascii=False,
            separators=(',', ':')).encode('utf-8')) if creating else 0
        start = time.perf_counter_ns()
        response = http_client.request('POST' if creating else 'GET', BASE_URL + path,
            json=payload if creating else None,
            headers={'Idempotency-Key': key} if creating else {}, timeout=BENCH_TIMEOUT_S)
        elapsed_ms = (time.perf_counter_ns() - start) / 1e6
        response.raise_for_status()
        data = response.json()
        assert data['id'] if creating else data['content'] == content
        response_bytes = len(response.content)
    return elapsed_ms, request_bytes, response_bytes


### Объяснение

**`measure_request()`** делает один вызов и возвращает задержку/размер запроса/размер ответа. Это заменяет четыре ветки внутри большого цикла. Create получает новый UUID, чтобы измерять вставку, а не идемпотентный возврат.

`perf_counter_ns()` измеряет длительность, деление на миллион даёт миллисекунды. Подготовка запроса вне таймера, SQLite внутри серверной работы. REST разбирает JSON после таймера, а gRPC к возврату уже разобрал сообщение: границы сохранены из исходной методики.

**Вопрос:** что за байты? Тела, не полный трафик: без заголовков, URL и фрейминга. UUID в gRPC-теле учитывается, в REST-заголовке — нет. REST Get имеет нулевое тело, но не нулевой сетевой запрос. Автоматических повторов в бенчмарке нет.


## 20. Серия замеров


In [ ]:
benchmark_rows, benchmark_samples = [], []
for size in BENCH_SIZES:
    content = 'x' * size
    for protocol in ('gRPC', 'REST'):
        for operation in ('Create', 'Get'):
            times, request_sizes, response_sizes = [], [], []
            failures = 0
            snippet_id = None
            if operation == 'Get':
                snippet_id = service.create(content, ttl_seconds=3600)['id']
            for index in range(BENCH_WARMUP + BENCH_REQUESTS_PER_CASE):
                measured = index >= BENCH_WARMUP
                try:
                    elapsed, request_size, response_size = measure_request(
                        protocol, operation, content, snippet_id)
                except Exception:
                    if not measured:
                        raise
                    failures += 1
                    continue
                if measured:
                    times.append(elapsed)
                    request_sizes.append(request_size)
                    response_sizes.append(response_size)
                    benchmark_samples.append(dict(protocol=protocol, operation=operation,
                        content_bytes=size, latency_ms=round(elapsed, 4)))
            row = summarize(protocol, operation, size, times, failures,
                round(statistics.mean(request_sizes), 1) if request_sizes else None,
                round(statistics.mean(response_sizes), 1) if response_sizes else None)
            benchmark_rows.append(row)
            print(protocol, operation, size, 'B | p95:', row['p95_ms'], 'ms | failures:', failures)
assert len(benchmark_rows) == 12
assert all(row['successes'] > 0 for row in benchmark_rows)


### Объяснение

Циклы отвечают только за порядок сценариев. Для Get запись создаётся заранее. Первые 10 вызовов — прогрев, остальные измеряются. Ошибка прогрева останавливает запуск; измеряемая ошибка увеличивает счётчик. Проверка в конце требует хотя бы один успех на сценарий, но не требует нуля ошибок.

**Вопрос:** это тест RPS? Нет, запросы последовательные. Фиксированный порядок и состояние базы влияют на числа; для общего вывода нужны повторные серии и реалистичная конкуренция.


## 21. CSV и читаемая таблица


In [ ]:
summary_path = WORKDIR / 'step5_benchmark_summary.csv'
samples_path = WORKDIR / 'step5_benchmark_samples.csv'
for path, rows in ((summary_path, benchmark_rows), (samples_path, benchmark_samples)):
    with path.open('w', encoding='utf-8', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

print(f'{"Protocol":8} {"Action":7} {"Bytes":>7} {"p50":>8} {"p95":>8} {"p99":>8} {"Errors":>7}')
for row in benchmark_rows:
    print(f'{row["protocol"]:8} {row["operation"]:7} {row["content_bytes"]:7} '
          f'{str(row["p50_ms"]):>8} {str(row["p95_ms"]):>8} {str(row["p99_ms"]):>8} {row["failures"]:7}')
print('Saved:', summary_path, samples_path)


### Объяснение

Сохраняется схема исходных CSV: summary содержит агрегаты, samples — отдельные успешные измерения. Каждый новый бенчмарк перезаписывает файлы только этой версии. Таблица показывает главное без pandas.

**Вопрос:** можно восстановить распределение по summary? Нет. Для него нужен samples; по трём перцентилям нельзя честно построить boxplot.


## 22. Относительное сравнение


In [ ]:
for operation in ('Create', 'Get'):
    for size in BENCH_SIZES:
        pair = {row['protocol']: row for row in benchmark_rows
                if row['operation'] == operation and row['content_bytes'] == size}
        g, r = pair['gRPC'], pair['REST']
        if g['p50_ms'] is not None and r['p50_ms']:
            reduction = 100 * (r['p50_ms'] - g['p50_ms']) / r['p50_ms']
            print(f'{operation} {size} B: снижение медианы у gRPC {reduction:+.1f}%')
        if g['response_bytes'] is not None and r['response_bytes']:
            reduction = 100 * (r['response_bytes'] - g['response_bytes']) / r['response_bytes']
            print(f'  уменьшение тела ответа {reduction:+.1f}%')


### Объяснение

Сравниваются пары одинаковых операций и размеров. Формула `(REST − gRPC) / REST × 100` показывает снижение показателя: плюс означает меньшее значение у gRPC, минус — у REST.

**Вопрос:** можно распространить процент на весь протокол? Нет: это конкретный запуск, размер, операция и показатель. Уменьшение тела ответа не равно уменьшению полного трафика.


## 23. Простые графики из CSV


In [ ]:
import matplotlib.pyplot as plt

with summary_path.open(encoding='utf-8', newline='') as file:
    chart_rows = list(csv.DictReader(file))
sizes = sorted({int(row['content_bytes']) for row in chart_rows})
fig, axes = plt.subplots(3, 2, figsize=(11, 10))
for row_index, metric in enumerate(('p50_ms', 'p95_ms', 'p99_ms')):
    for column, operation in enumerate(('Create', 'Get')):
        ax = axes[row_index, column]
        for offset, protocol, color in ((-0.18, 'gRPC', '#237B83'), (0.18, 'REST', '#DB7934')):
            selected = sorted((row for row in chart_rows
                if row['operation'] == operation and row['protocol'] == protocol),
                key=lambda row: int(row['content_bytes']))
            values = [float(row[metric]) if row[metric] else math.nan for row in selected]
            bars = ax.bar([i + offset for i in range(len(sizes))], values,
                          width=0.36, label=protocol, color=color)
            ax.bar_label(bars, fmt='%.3f', fontsize=8, padding=3)
        ax.set_xticks(range(len(sizes)), [f'{size} B' for size in sizes])
        ax.set(title=f'{operation}: {metric}', ylabel='Задержка, мс (меньше — лучше)')
        ax.margins(y=0.2)
        ax.legend()
fig.suptitle('gRPC и REST: последовательные запросы на localhost')
failures = sum(int(row['failures']) for row in chart_rows)
fig.text(0.5, 0.01, f'Ошибок: {failures}. Задержки успешных вызовов; p99 чувствителен к малой выборке.',
         ha='center', fontsize=9)
fig.tight_layout(rect=(0, 0.04, 1, 0.96))
figure_path = WORKDIR / 'step5_latency.png'
fig.savefig(figure_path, dpi=150)
plt.show()
plt.close(fig)
print('Saved:', figure_path)


### Объяснение

Один короткий блок рисует p50/p95/p99 для Create/Get. CSV читается с диска, значения подписываются, PNG сохраняется рядом. PDF и дополнительные панели оформления убраны; размеры сообщений остались в CSV.

**Вопрос:** можно сравнивать высоты разных панелей? Шкалы независимы: сравнивай пары внутри панели и числовые подписи. Хорошая медиана не исключает плохого p99. Для графика без серверов достаточно Matplotlib, импортов `csv`/`math` и пути `summary_path` к готовому CSV.


## 24. Завершение


In [ ]:
if STOP_AT_END:
    stop_servers()
    print('Серверы остановлены.')
else:
    print('Серверы оставлены для браузера:', BASE_URL + '/docs')
    print('По окончании вызови stop_servers()')


### Объяснение

По умолчанию полный запуск останавливает серверы. Для браузера заранее поставь `STOP_AT_END=False`. После ошибки в середине выполни `stop_servers()`, если функция уже объявлена, либо перезапусти ядро.

**Вопрос:** остановка удаляет базу? Нет, файлы остаются. Закрытие Safari не выключает сервер; перезапуск ядра останавливает серверы его процесса, но не внешнего терминала.


## Самое важное для защиты

- **`_validate()`, `create()`, `get()`**: входные правила, транзакция, идемпотентность, срок доступности.
- **REST-обработчики и `CreateSnippet`/`GetSnippet`**: два транспорта, общий сервис, разные форматы ошибок.
- **`call_with_retries()`**: общий бюджет, backoff/jitter и один ключ на попытки. Дедлайн не доказывает откат сохранения.
- **`measure_request()`, `percentile()`, `summarize()`**: границы таймера, прогрев, перцентили успешных вызовов, ошибки и размеры тел.

Возможные вопросы: «Что будет, если ответ потерялся после сохранения?», «Почему 422 не повторяется?», «Почему после TTL строка есть, но не читается?», «Что даёт и чего не доказывает локальный бенчмарк?», «Почему используется Python при требовании Java?».

Для демонстрации: создай свой текст через `/docs`, открой URL, покажи повтор с одним ключом и конфликт 409, объясни одну пару столбцов p95 и ограничения замера. Предполагаемые вопросы не заменяют фактические требования преподавателя.


## Гайд: создать текст и получить ссылку

1. Выставь `STOP_AT_END=False` и выполни ячейки до запуска серверов включительно. Для своего текста бенчмарк не нужен. Не запускай одновременно исходный Step 5.
2. Открой [http://127.0.0.1:18080/docs](http://127.0.0.1:18080/docs). Раскрой **POST /v1/snippets → Try it out**. Необязательный `idempotency-key` можно оставить пустым.
3. В Request body введи:

```json
{
  "content": "Мой текст\nВторая строка",
  "title": "Пример",
  "language": "plaintext",
  "ttl_seconds": 3600
}
```

4. Нажми **Execute**. В фактическом Server response ожидается 201. Скопируй `url` и открой в Safari — появится текст. Сохрани ID/ссылку: общего списка записей нет.
5. Для метаданных используй **GET /v1/snippets/{snippet_id}**: введи только ID, не весь URL. **GET /s/{snippet_id}** возвращает только текст.
6. Без ключа каждый POST создаёт новую запись. Для безопасного повтора перед первым запросом задай UUID, сохрани тот же ключ и все поля. Изменение данных с прежним ключом даёт 409. UUID можно получить через `uuid.uuid4()` в Python.
7. TTL: 60 секунд–30 дней, без поля — семь дней. Текст: до 1 МиБ UTF-8. Редактирования нет: исправленный текст создаётся заново. Повтор с ключом не продлевает срок. После TTL — 404.
8. Для остановки вызови `stop_servers()` или перезапусти ядро. Если `STOP_AT_END=True`, полный Run All уже выключил серверы. Полный повторный запуск начинай с чистого ядра.

Нет соединения — сервер не запущен/остановлен. 404 на `/` — открой `/docs`; 404 на записи — неверный ID, истёк TTL или другая база. 422 — проверь данные, 413 — большой текст, 409 — конфликт ключа. Занятый порт — останови предыдущий запуск. gRPC-порт 50071 не открывается как сайт.

`127.0.0.1` доступен только на твоём компьютере; отправка такой ссылки другу не публикует сервер. База этой версии отдельная, поэтому ID исходной версии здесь не появятся автоматически.

Результаты: [база](./sis2_pastebin_runtime/simplified/pastebin.sqlite3), [summary CSV](./sis2_pastebin_runtime/simplified/step5_benchmark_summary.csv), [samples CSV](./sis2_pastebin_runtime/simplified/step5_benchmark_samples.csv), [график](./sis2_pastebin_runtime/simplified/step5_latency.png). База сохраняется между запусками; результаты бенчмарка перезаписываются.
